# Simple Hessian Factor Comparison

Compare EKFAC factors (covariances, eigenvectors, eigenvalue corrections) between bergson and kronfluence.

Model: pythia-14m-deduped, plain text dataset.

In [1]:
from pathlib import Path

import torch
from safetensors.torch import load_file

In [2]:
def load_hessian_results(head_dir, file_map, device="cuda"):
    """Load hessian results from a directory.

    Handles both single-file (e.g. kronfluence .safetensors) and
    sharded subdirectory (e.g. bergson shard_*.safetensors) formats.
    """
    results = {}
    head_dir = Path(head_dir)
    for key, filename in file_map.items():
        path = head_dir / filename
        if path.is_file():
            results[key] = load_file(str(path), device=device)
        elif path.is_dir():
            shards = sorted(path.glob("shard_*.safetensors"))
            if not shards:
                print(f"  WARNING: no shards found in {path}")
                continue
            shard_data = [load_file(str(s), device=device) for s in shards]
            results[key] = {
                k: torch.cat([d[k] for d in shard_data], dim=0) for k in shard_data[0]
            }
        else:
            print(f"  WARNING: {path} not found")
    return results


def strip_prefix(key):
    """Strip the first prefix segment (everything before the first '.')."""
    return key.split(".", 1)[1] if "." in key else key


def match_keys(dict_a, dict_b):
    """Find matching keys between two dicts, handling prefix differences."""
    matches = []
    used_b = set()
    for key_a in dict_a:
        if key_a in dict_b:
            matches.append((key_a, key_a, key_a))
            used_b.add(key_a)
            continue
        stripped_a = strip_prefix(key_a)
        if stripped_a in dict_b:
            matches.append((key_a, stripped_a, stripped_a))
            used_b.add(stripped_a)
            continue
        for key_b in dict_b:
            if key_b not in used_b and strip_prefix(key_b) == stripped_a:
                matches.append((key_a, key_b, stripped_a))
                used_b.add(key_b)
                break
    return matches


def compare_covariances(
    results_a, results_b, label_a="A", label_b="B", categories=None
):
    if categories is None:
        categories = ["grad_cov", "act_cov"]
    print(f"\n=== Covariance Comparison ({label_a} vs {label_b}) ===")
    for category in categories:
        print(f"\n--- {category} ---")
        dict_a = results_a[category]
        dict_b = results_b[category]
        for key_a, key_b, display in match_keys(dict_a, dict_b):
            af, bf = dict_a[key_a].float(), dict_b[key_b].float()
            if af.shape != bf.shape:
                print(
                    f"  {display}: SHAPE MISMATCH {label_a}={list(af.shape)} {label_b}={list(bf.shape)}"
                )
                continue
            cosine_sim = (
                (af / (af.norm() + 1e-12) * bf / (bf.norm() + 1e-12)).sum().item()
            )
            allclose = torch.allclose(af, bf, rtol=1e-3, atol=1e-6)
            if not allclose:
                diff = (af - bf).abs()
                rel_diff = diff / (af.abs() + 1e-12)
                max_rel_diff = rel_diff.max()
                print(
                    f"  {display}: cosine={cosine_sim:.6f}, allclose(rtol=1e-3)={allclose}, max_rel_diff={max_rel_diff:.6f}"
                )
            else:
                print(
                    f"  {display}: cosine={cosine_sim:.6f}, allclose(rtol=1e-3)={allclose}"
                )


def compare_eigenvectors(
    results_a, results_b, label_a="A", label_b="B", categories=None
):
    if categories is None:
        categories = ["grad_eigenvectors", "act_eigenvectors"]
    print(f"\n=== Eigenvector Comparison ({label_a} vs {label_b}, sign-invariant) ===")
    for category in categories:
        print(f"\n--- {category} ---")
        dict_a = results_a[category]
        dict_b = results_b[category]
        for key_a, key_b, display in match_keys(dict_a, dict_b):
            af, bf = dict_a[key_a].float(), dict_b[key_b].float()
            if af.shape != bf.shape:
                print(
                    f"  {display}: SHAPE MISMATCH {label_a}={list(af.shape)} {label_b}={list(bf.shape)}"
                )
                continue
            cosines = (af * bf).sum(dim=-1) / (
                af.norm(dim=-1) * bf.norm(dim=-1) + 1e-12
            )
            print(
                f"  {display}: mean|cos|={cosines.abs().mean():.6f}, min|cos|={cosines.abs().min():.6f}"
            )


def compare_lambdas(results_a, results_b, label_a="A", label_b="B"):
    print(f"\n=== Lambda / Eigenvalue Correction ({label_a} vs {label_b}) ===")
    dict_a = results_a["lambda"]
    dict_b = results_b["lambda"]
    for key_a, key_b, display in match_keys(dict_a, dict_b):
        af, bf = dict_a[key_a].float().flatten(), dict_b[key_b].float().flatten()
        if af.shape != bf.shape:
            print(
                f"  {display}: SHAPE MISMATCH {label_a}={list(af.shape)} {label_b}={list(bf.shape)}"
            )
            continue
        corr = torch.corrcoef(torch.stack([af, bf]))[0, 1].item()
        mask = (bf.abs() > 1e-10) & (af.abs() > 1e-10)
        median_ratio = (
            (af[mask] / bf[mask]).median().item() if mask.sum() > 0 else float("nan")
        )
        print(
            f"  {display}: corr={corr:.6f}, median_ratio({label_a}/{label_b})={median_ratio:.4f}"
        )

In [ ]:
# ── File maps for different result formats ───────────────────────────────────
KRONFLUENCE_FILE_MAP = {
    "grad_cov": "gradient_covariance.safetensors",
    "grad_eigenvectors": "gradient_eigenvectors.safetensors",
    "act_cov": "activation_covariance.safetensors",
    "act_eigenvectors": "activation_eigenvectors.safetensors",
    "lambda": "lambda_matrix.safetensors",
}

BERGSON_FILE_MAP = {
    "grad_eigenvectors": "eigen_gradient_sharded",
    "act_eigenvectors": "eigen_activation_sharded",
    "lambda": "eigenvalue_correction_sharded",
    "act_cov": "activation_sharded",
    "grad_cov": "gradient_sharded",
}

# Works whether notebook runs from repo root or from hessian_simple/
RESULTS = Path("results")
if not RESULTS.is_dir():
    RESULTS = Path("hessian_simple/results")

path_kron = RESULTS / "kronfluence" / "factors_ekfac"
path_berg = RESULTS / "bergson" / "kfac"

results_kron = load_hessian_results(path_kron, KRONFLUENCE_FILE_MAP)
results_berg = load_hessian_results(path_berg, BERGSON_FILE_MAP)
print(f"Kronfluence keys: {list(results_kron.keys())}")
print(f"Bergson keys: {list(results_berg.keys())}")

In [ ]:
compare_covariances(results_kron, results_berg, label_a="kron", label_b="berg")
compare_eigenvectors(results_kron, results_berg, label_a="kron", label_b="berg")
compare_lambdas(results_kron, results_berg, label_a="kron", label_b="berg")